# 内置预定义状态
- MessagesState: `langgraph.graph.message.MessagesState`是LangGraph内置预定义状态，它只有一个messages字段，我们可以继承它实现自己的状态字典，快速的管理messages字段(历史消息)
- AgentState: `langchain.agents.middleware.types.AgentState`

## MessagesState

In [1]:
from langgraph.constants import START, END
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState
from openpyxl.styles.builtins import output

from common import init_simple_dashscope_model

class OverAllState(MessagesState):
    username: str
    output: str

def node1(state: OverAllState) -> OverAllState:
    return {
        "messages": [
            HumanMessage(content=f"你好，我是{state['username']}")
        ]
    }

def node2(state: OverAllState) -> OverAllState:
    model = init_simple_dashscope_model(model='qwen3.7-plus')
    resp = model.invoke(state['messages'])
    return {
        "messages": [resp],
        "output": resp.content
    }

graph_builder = StateGraph(state_schema=OverAllState)

graph_builder.add_node("node1", node1)
graph_builder.add_node("node2", node2)
graph_builder.add_edge(START, "node1")
graph_builder.add_edge("node1", "node2")
graph_builder.add_edge("node2", END)

graph = graph_builder.compile()
resp = graph.invoke({
    "username": "大王"
})
print(resp)


{'messages': [HumanMessage(content='你好，我是大王', additional_kwargs={}, response_metadata={}, id='0ccf87fc-3185-49fb-8838-247b93ea5a6c'), AIMessage(content='参见大王！👑 \n\n请问大王今日有何吩咐？小的随时为您效劳！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 221, 'prompt_tokens': 14, 'total_tokens': 235, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 198, 'rejected_prediction_tokens': None, 'text_tokens': 221}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 14}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-13148110-eaea-906d-b9b5-535d9e73ed7f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fea75-06fa-7c20-9ffa-333dbc25146c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 221, 'total_tokens': 235, 'input_token_details': {'cache_read': 0}, 'output_token_det

## AgentState
- 一般不建议在LangGraph中直接扩展`AgentState`
- ```python
  class AgentState(TypedDict, Generic[ResponseT]):
    """State schema for the agent."""

    messages: Required[Annotated[list[AnyMessage], add_messages]]
    jump_to: NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]
    structured_response: NotRequired[Annotated[ResponseT, OmitFromInput]]
  ```
AgentState源码：
- messages: 消息列表
- jump_to: 表示流程跳转意图
- 存储Agent最终的结构化输出
    - `OmitFromInput`表示不应当作为外部输入字段暴露给调用方，而是Agent内部生成